In [1]:
import requests
from json import dumps # для красивого вывода json
import pandas as pd
import fastparquet
#import apimoex
#import pyarrow

### Задание

Необходимо загрузить данные и создать все сопутствующие для этого объекты для реализации витрины (внутренняя таблица в GreenPlum) следующего вида:

Найти самые крупные сделки для конкретной акции для каждого типа (BUY / SELL) в разрезе дня
|Наименование поля|Описание|
| :---: | :---: |
|BOARDID |ID режима торгов|
|BOARD_NAME|Наименование режима торгов|
|ISIN|ISIN код|
|ISQUALIFIEDINVESTORS|Признак предназачена ли бумага для квалифицированных инвесторов|
|SECID|ID акции|
|PRICE|Цена|
|VALUE|Сумма сделки|
|QUANTITY|Кол-во|
|DEAL_TYPE|Тип сделки (BUY / SELL)|
|DEAL_TIME|Время осуществления сделки|
|TRADE_SESSION_DATE|Дата сессии|

### Решение

Согласно источникам
- [Реестр сделок](https://www.moex.com/ru/marketdata/archive/#/engine=stock&market=shares&data_type=trades&data_interval=current&data_format=csv&year=2026)
- [Информационные продукты MOEX](https://www.moex.com/ru/orders?realtime)
- [Итоги торгов](https://www.moex.com/ru/marketdata/#/mode=groups&group=4&collection=3&boardgroup=57&data_type=history&date=2026-06-03&category=main)

скачивание данных о сделках с вэб-сайта MOEX возможно только по платной подписке. При этом, можно ознакомиться с содержание данных о сделках в документации:

- [Описание состава и формата данных](https://www.moex.com/ru/documents/opisanie-sostava-i-formata-dannyh)

Согласно источникам

- [Инструменты фондового рынка](https://www.moex.com/msn/stock-instruments#/)
- [Таблица соответствия торгуемых ценных бумаг по режимам торгов](https://www.moex.com/ru/SecuritiesListing.aspx)

акции могут торговаться в следующих режимах

|PRIMARY_BOARDID|PRIMARY_BOARD_TITLE|
| :---: | :---: |
|TQBR|Т+: Акции и ДР - безадрес.|
|MTQR|ОТС: Акции Т+ - безадресные|
|MPTR|ОТС: Акции РПС с ЦК - адресные|
|RPEY|РЕПО в ин.валюте (CNY) - адрес.|
|LIQR|РЕПО с ЦК: Урегулирование - безадрес.|
|FQBR|Т+ Ин.Акции и ДР|
|FSEQ|РПС: Ин.Акции|
|FTEQ|РПС с ЦК: Ин.Акции и ДР|
|PSEQ|РПС : Акции|
|PTEQ|РПС с ЦК: Акции и ДР|
|SPEQ|Поставка по СК (акции)|
|TQDP|Крупные пакеты - Акции|
|PSPI|РПС Акции ПИР|
|PTPI|РПС с ЦК Акции ПИР|
|TQPI|Т+ Акции ПИР|
|PTSD|РПС с ЦК: Акции и ДР(расч.USD)|


Согласно источнику [Orders of "Book or cancel" type on the stock market](https://www.moex.com/n52316?nt=209), существует еще ряд режимов торгов акциями:

|PRIMARY_BOARDID|PRIMARY_BOARD_TITLE|
| :---: | :---: |
|SMAL|T+: Odd lot trading|
|TQBD|T+: FRGN stocks, DRs USD|
|TQBE|T+: Shares and DRs EUR|
|TQIF|T+: investment funds|
|TQTC|T+: ETC|
|TQTD|T+: ETF USD|
|TQTE|T+: ETF EUR|
|TQTF|T+: ETF|
|FQDE|T+: inv. risk stocks FRGN|
|TQFD|T+: Inv. funds USD|
|TQFE|T+: Inv. funds EUR|
|TQPD|T+ stocks inv. Risk USD|
|TQPE|T+ stocks inv. Risk EUR|
|TQDD|T+: inv. risk stocks FRGN USD|

Перечисленные режимы официально существуют, но их функциональность в текущих условиях санкций на биржу, клиринговый центр и НРД сильно ограничена или полностью парализована.

Важно отметить, что secid со временем может меняться [Информация по изменению торговых кодов бумаг](https://www.moex.com/ru/SecuritiesListing.aspx).

Для получения данных о сделках используем [MOEX ISS API](https://iss.moex.com/iss/reference/), а именно [Все сделки рынка](https://iss.moex.com/iss/reference/379).
Используем параметр `date` - для выгрузки истории сделок за нужный торговый день (2026-06-04). Максимальное значение выгружаемых строк = 5000. Сделки возвращаются в порядке их заключения. Напишем цикл для извлечения всех сделок торгового дня (с использованием параметр `tradeno` - номер сделки, с которого следует начать возвращать данные.). 


In [2]:
# считываем первые 5000 строк
res = requests.get('https://iss.moex.com//iss/history/engines/stock/markets/shares/trades.json?date=2026-06-04')
df_res = pd.DataFrame(res.json()["trades"]["data"], columns = res.json()["trades"]["columns"])
print(df_res.head(1))
print(df_res.tail(1))
print(df_res.shape[0])

# определяем номер последней сделки в считанных данных
last_tradeno = df_res.loc[4999, 'TRADENO']
print(last_tradeno)

i=0
df = df_res
while df.shape[0] == 5000:
    # считываем следующие 5000 строк
    res = requests.get(f'https://iss.moex.com//iss/history/engines/stock/markets/shares/trades.json?date=2026-06-04&tradeno={last_tradeno}')
    df = pd.DataFrame(res.json()["trades"]["data"], columns = res.json()["trades"]["columns"])
    i += 1
    # определяем номер последней сделки
    last_tradeno = df.loc[df.shape[0]-1, 'TRADENO']
    # добавляем новые сделки к результирующему датафрейму
    df_res = pd.concat([df_res, df])
    # проверка
    print(i, df.loc[0, 'TRADENO'], last_tradeno, df.shape[0], df_res.shape[0], df.loc[0, 'TRADETIME'], df.loc[df.shape[0]-1, 'TRADETIME'])

       TRADENO   TRADEDATE TRADETIME SECID BOARDID   PRICE  QUANTITY  VALUE  \
0  16659452650  2026-06-04  06:53:13  FIXR    SMAL  0.5094       158  80.49   

   TYPE BUYSELL  TRADINGSESSION TRADE_SESSION_DATE  
0  None       B               0         2026-06-04  
          TRADENO   TRADEDATE TRADETIME SECID BOARDID  PRICE  QUANTITY  \
4999  16659458668  2026-06-04  07:00:00  ROSN    TQBR  396.5         3   

       VALUE  TYPE BUYSELL  TRADINGSESSION TRADE_SESSION_DATE  
4999  1189.5  None       B               0         2026-06-04  
5000
16659458668
1 16659458669 16659464684 5000 10000 07:00:00 07:01:57
2 16659464685 16659470238 5000 15000 07:01:57 07:04:24
3 16659470240 16659475655 5000 20000 07:04:25 07:06:41
4 16659475656 16659481292 5000 25000 07:06:41 07:09:15
5 16659481293 16659487039 5000 30000 07:09:15 07:13:12
6 16659487040 16659493083 5000 35000 07:13:12 07:18:13
7 16659493084 16659500632 5000 40000 07:18:13 07:21:49
8 16659500633 16659506223 5000 45000 07:21:49 07:25:02
9

In [3]:
# проверка на дубликаты
print(df_res.shape[0])
print(len(df_res['TRADENO'].unique()))

2641370
2641370


In [4]:
# содержимое поля TRADETIME
df_res.groupby('TRADINGSESSION').agg({'TRADETIME': ['min', 'max']})

TRADETIME          
                     min       max
TRADINGSESSION                    
0               06:53:13  09:49:58
1               09:50:00  18:59:29
2               19:00:01  23:49:58

In [5]:
# содержимое поля TRADEDATE, TRADE_SESSION_DATE
print(df_res['TRADEDATE'].unique())
print(df_res['TRADE_SESSION_DATE'].unique())

<ArrowStringArray>
['2026-06-04']
Length: 1, dtype: str
<ArrowStringArray>
['2026-06-04']
Length: 1, dtype: str


In [6]:
# отобразим первые строки получившегося датафрейма
display(df_res.head(2))

,TRADENO,TRADEDATE,TRADETIME,SECID,BOARDID,PRICE,QUANTITY,VALUE,TYPE,BUYSELL,TRADINGSESSION,TRADE_SESSION_DATE
0,16659452650,2026-06-04,06:53:13,FIXR,SMAL,0.5094,158,80.49,None,B,0,2026-06-04
1,16659452653,2026-06-04,06:58:14,FIXR,SMAL,0.5234,1,0.52,None,B,0,2026-06-04


In [7]:
# сохраним только нужные колонки
df_res = df_res[['TRADENO', 'TRADE_SESSION_DATE', 'TRADETIME', 'SECID', 'BOARDID', 'PRICE', 'QUANTITY', 'VALUE', 'BUYSELL']]

# отобразим первые строки получившегося датафрейма
display(df_res.head(2))

,TRADENO,TRADE_SESSION_DATE,TRADETIME,SECID,BOARDID,PRICE,QUANTITY,VALUE,BUYSELL
0,16659452650,2026-06-04,06:53:13,FIXR,SMAL,0.5094,158,80.49,B
1,16659452653,2026-06-04,06:58:14,FIXR,SMAL,0.5234,1,0.52,B


In [8]:
# переведем название столбцов в нижний регистр
new_col_name = [v.lower() for v in df_res.columns]
df_res.columns = new_col_name
# отобразим первые строки получившегося датафрейма
display(df_res.head(2))

,tradeno,trade_session_date,tradetime,secid,boardid,price,quantity,value,buysell
0,16659452650,2026-06-04,06:53:13,FIXR,SMAL,0.5094,158,80.49,B
1,16659452653,2026-06-04,06:58:14,FIXR,SMAL,0.5234,1,0.52,B


Сопоствим поля из выгрузки и требуемые по заданию:
|Наименование поля|Описание|В выгрузке|
| :---: | :---: | :---: |
|BOARDID |ID режима торгов|BOARDID|
|BOARD_NAME|Наименование режима торгов||
|ISIN|ISIN код||
|ISQUALIFIEDINVESTORS|Признак предназачена ли бумага для квалифицированных инвесторов||
|SECID|ID акции|SECID|
|PRICE|Цена|PRICE|
|VALUE|Сумма сделки|VALUE|
|QUANTITY|Кол-во|QUANTITY|
|DEAL_TYPE|Тип сделки (BUY / SELL)|BUYSELL|
|DEAL_TIME|Время осуществления сделки|TRADETIME|
|TRADE_SESSION_DATE|Дата сессии|TRADE_SESSION_DATE|

Дополнительно нужно найти [BOARD_NAME](https://iss.moex.com/iss/reference/351), ISIN, ISQUALIFIEDINVESTORS. ISIN, ISQUALIFIEDINVESTORS возьмем с сайта [Список ценных бумаг, допущенных к торгам по состоянию на...](https://www.moex.com/ru/listing/securities-list.aspx), скачаем листинг в csv и сохраним в \data\Export_ru_securities-list_20260605.csv.

In [9]:
# сбор наименований режимов торгов
res_board_name = requests.get('https://iss.moex.com/iss/engines/stock/markets/shares.json?date=2026-06-04')
# print(dumps(res.json(), indent=4, ensure_ascii=False))
df_board_name = pd.DataFrame(res_board_name.json()["boards"]["data"], columns = res_board_name.json()["boards"]["columns"])
display(df_board_name.head(2))
print(df_board_name.shape[0])

,id,board_group_id,boardid,title,is_traded
0,1,6,EQBR,Основной режим: А1-Акции и паи - безадрес.,0
1,2,6,EQBS,Основной режим: А2-Акции и паи - безадрес.,0


35


In [10]:
# сохраним только нужные колонки
df_board_name =  df_board_name[['boardid', 'title']]
display(df_board_name.head(2))

,boardid,title
0,EQBR,Основной режим: А1-Акции и паи - безадрес.
1,EQBS,Основной режим: А2-Акции и паи - безадрес.


In [11]:
# сбор isin, isqulifiedinvestors
df_sec = pd.read_csv('data/Export_ru_securities-list_20260605.csv', sep=',', encoding='ANSI')
display(df_sec.head(2))

,DATESTAMP,INSTRUMENT_ID,LIST_SECTION,NPP,SUPERTYPE,INSTRUMENT_TYPE,INSTRUMENT_CATEGORY,TRADE_CODE,ISIN,REGISTRY_NUMBER,...,OTHER_SECURITIES,DISCLOSURE_PART_PAGE,DISCLOSURE_RF_INFO_PAGE,INCLUDE_DATE,CFI_FOREIGN,ISIN_UNDERLYING_ASSET,CFI_UNDERLYING_ASSET,PIF_STATUS,PIF_STATUS_HIST,OBLIGATION_PROGRAM_DATE
0,04.06.2026 00:00:00,15348,Первый уровень,1,Облигации,Облигация биржевая,биржевые облигации процентные неконвертируемые...,NaN,NaN,4B02-02-00354-B-003P,...,"Облигация корпоративная, 41900354B, RU000A0ZZF...",https://www.gazprombank.ru,https://www.e-disclosure.ru/portal/company.asp...,04.06.2021 00:00:00,NaN,NaN,NaN,NaN,NaN,27.08.2018 00:00:00
1,04.06.2026 00:00:00,15349,Первый уровень,2,Облигации,Облигация биржевая,биржевые облигации процентные неконвертируемые...,NaN,NaN,4B02-03-00354-B-003P,...,"Облигация корпоративная, 41900354B, RU000A0ZZF...",https://www.gazprombank.ru,https://www.e-disclosure.ru/portal/company.asp...,04.06.2021 00:00:00,NaN,NaN,NaN,NaN,NaN,27.08.2018 00:00:00


In [12]:
# сохраним только нужные колонки
df_sec = df_sec[['TRADE_CODE', 'ISIN', 'SUPERTYPE', 'QUALIFIED_INVESTOR']] #[df_sec['SUPERTYPE'] == 'Акции']
#переведем название столбцов в нижний регистр
new_col_name = [v.lower() for v in df_sec.columns]
df_sec.columns = new_col_name
# отобразим первые строки получившегося датафрейма
display(df_sec.head(2))


,trade_code,isin,supertype,qualified_investor
0,NaN,NaN,Облигации,NaN
1,NaN,NaN,Облигации,NaN


In [13]:
#объеденим датафреймы
df_res = df_res.merge(df_board_name, on='boardid', how='left')
df_res = df_res.merge(df_sec, left_on ='secid', right_on='trade_code', how='left')
display(df_res.head(2))
display(df_res.shape[0])

,tradeno,trade_session_date,tradetime,secid,boardid,price,quantity,value,buysell,title,trade_code,isin,supertype,qualified_investor
0,16659452650,2026-06-04,06:53:13,FIXR,SMAL,0.5094,158,80.49,B,Т+: Неполные лоты (акции) - безадрес.,FIXR,RU000A10B5G8,Акции,NaN
1,16659452653,2026-06-04,06:58:14,FIXR,SMAL,0.5234,1,0.52,B,Т+: Неполные лоты (акции) - безадрес.,FIXR,RU000A10B5G8,Акции,NaN


2641370

In [14]:
df_res[df_res['title'].isnull()]

,tradeno,trade_session_date,tradetime,secid,boardid,price,quantity,value,buysell,title,trade_code,isin,supertype,qualified_investor


In [15]:
df_res[df_res['isin'].isnull()]

,tradeno,trade_session_date,tradetime,secid,boardid,price,quantity,value,buysell,title,trade_code,isin,supertype,qualified_investor


In [16]:
df_res['supertype'].value_counts()

supertype
Акции                    1758041
Инвестиционные паи        883036
Депозитарные расписки        293
Name: count, dtype: int64

In [17]:
# отфильтруем акции
df_res = df_res[df_res['supertype'] == 'Акции']
# оставим только нужные столбцы
df_res = df_res[['tradeno', 'trade_session_date', 'tradetime', 'secid', 'boardid', 'price', 'quantity', 'value', 'buysell', 'title', 'isin', 'qualified_investor']]
# приведем дату к типу datetime
# df_res['trade_session_date'] = pd.to_datetime(df_res['trade_session_date'])
# отобразим первые строки получившегося датафрейма
display(df_res.head(2))
display(df_res.shape[0])

,tradeno,trade_session_date,tradetime,secid,boardid,price,quantity,value,buysell,title,isin,qualified_investor
0,16659452650,2026-06-04,06:53:13,FIXR,SMAL,0.5094,158,80.49,B,Т+: Неполные лоты (акции) - безадрес.,RU000A10B5G8,NaN
1,16659452653,2026-06-04,06:58:14,FIXR,SMAL,0.5234,1,0.52,B,Т+: Неполные лоты (акции) - безадрес.,RU000A10B5G8,NaN


1758041

In [24]:
len(df_res['tradeno'].unique())

1758041

In [18]:
df_res['boardid'].value_counts()

boardid
TQBR    1754116
SMAL       3925
Name: count, dtype: int64

In [19]:
df_res['buysell'].value_counts()

buysell
B    892339
S    865702
Name: count, dtype: int64

In [20]:
df_res.info()

<class 'pandas.DataFrame'>
Index: 1758041 entries, 0 to 2641369
Data columns (total 12 columns):
 #   Column              Dtype  
---  ------              -----  
 0   tradeno             int64  
 1   trade_session_date  str    
 2   tradetime           str    
 3   secid               str    
 4   boardid             str    
 5   price               float64
 6   quantity            int64  
 7   value               float64
 8   buysell             str    
 9   title               str    
 10  isin                str    
 11  qualified_investor  str    
dtypes: float64(2), int64(2), str(8)
memory usage: 313.4 MB


In [ ]:
#df_res = df_res.reset_index()
df_res = df_res.set_index("tradeno")


In [39]:
display(df_res.head(2))

,trade_session_date,tradetime,secid,boardid,price,quantity,value,buysell,title,isin,qualified_investor
tradeno,,,,,,,,,,,
16659452650,2026-06-04,06:53:13,FIXR,SMAL,0.5094,158,80.49,B,Т+: Неполные лоты (акции) - безадрес.,RU000A10B5G8,NaN
16659452653,2026-06-04,06:58:14,FIXR,SMAL,0.5234,1,0.52,B,Т+: Неполные лоты (акции) - безадрес.,RU000A10B5G8,NaN


In [43]:
df_res.columns

Index(['trade_session_date', 'tradetime', 'secid', 'boardid', 'price',
       'quantity', 'value', 'buysell', 'title', 'isin', 'qualified_investor'],
      dtype='str')

In [44]:
df_res.to_parquet("data/stock_market_securities_20260604.parquet", compression = 'snappy', engine='fastparquet')

In [45]:
# отобразим список тикеров начинающихся на I
sorted(df_res[df_res['secid'].apply(lambda x: x[:1])=="I"]['secid'].unique())

['IGST', 'IGSTP', 'IRAO', 'IRKT', 'IVAT']